In [1]:
# Данный ноутбук использовал окружение google-colab
%pip install catboost fasttext -q

Note: you may need to restart the kernel to use updated packages.


# Домашнее задание "NLP. Часть 1"

In [2]:
import math
import re
import os
import random
import json
from collections import Counter, defaultdict
from typing import List, Dict, Tuple, Any

import torch
import numpy as np
import datasets
import fasttext
import fasttext.util
from transformers import BertTokenizer, BertModel

/home/Applications/Data/programming/2025/data-portfolio/skils/NN_skils/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def seed_everything(seed: int):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

seed_everything(42)

In [4]:
def normalize_pretokenize_text(text: str) -> List[str]:
    text = text.lower()
    words = re.findall(r'\b\w+\b', text)
    return words

In [5]:
# This block is for tests only
test_corpus = [
    "the quick brown fox jumps over the lazy dog",
    "never jump over the lazy dog quickly",
    "brown foxes are quick and dogs are lazy"
]

def build_vocab(texts: List[str]) -> Tuple[List[str], Dict[str, int]]:
    all_words = []
    for text in texts:
        words = normalize_pretokenize_text(text)
        all_words.extend(words)
    vocab = sorted(set(all_words))
    vocab_index = {word: idx for idx, word in enumerate(vocab)}
    return vocab, vocab_index

vocab, vocab_index = build_vocab(test_corpus)

In [6]:
vocab_index

{'and': 0,
 'are': 1,
 'brown': 2,
 'dog': 3,
 'dogs': 4,
 'fox': 5,
 'foxes': 6,
 'jump': 7,
 'jumps': 8,
 'lazy': 9,
 'never': 10,
 'over': 11,
 'quick': 12,
 'quickly': 13,
 'the': 14}

## Задание 1 (0.5 балла)
Реализовать One-Hot векторизацию текстов

In [7]:
def one_hot_vectorization(text: str, vocab: List[str] = None, vocab_index: Dict[str, int] = None) -> List[int]:
    words = normalize_pretokenize_text(text)
    one_hot_matrix = [0] * len(vocab)
    for word in words:
        one_hot_matrix[vocab_index[word]] = 1
    return one_hot_matrix

def test_one_hot_vectorization(corpus, vocab, vocab_index) -> bool:
    try:
        text = "the quick brown fox"
        result = one_hot_vectorization(text, vocab, vocab_index)
        if not isinstance(result, list):
            return False
        expected_length = len(vocab)
        if len(result) != expected_length:
            return False

        words_in_text = normalize_pretokenize_text(text)
        for word in words_in_text:
            if word in vocab_index:
                idx = vocab_index[word]
                if result[idx] != 1:
                    return False

        print("One-Hot-Vectors test PASSED")

        return True
    except Exception as e:
        print(f"One-Hot-Vectors test FAILED: {e}")
        return False

In [8]:
assert test_one_hot_vectorization(test_corpus, vocab, vocab_index)

One-Hot-Vectors test PASSED


## Задание 2 (0.5 балла)
Реализовать Bag-of-Words

In [9]:
def bag_of_words_vectorization(text: str) -> Dict[str, int]:
    return Counter(normalize_pretokenize_text(text))

def test_bag_of_words_vectorization() -> bool:
    try:
        text = "the the quick brown brown brown"
        result = bag_of_words_vectorization(text)

        if not isinstance(result, dict):
            return False

        if result.get('the', 0) != 2:
            return False
        if result.get('quick', 0) != 1:
            return False
        if result.get('brown', 0) != 3:
            return False
        if result.get('nonexistent', 0) != 0:
            return False

        print("Bad-of-Words test PASSED")
        return True
    except Exception as e:
        print(f"Bag-of-Words test FAILED: {e}")
        return False

In [10]:
assert test_bag_of_words_vectorization()

Bad-of-Words test PASSED


## Задание 3 (0.5 балла)
Реализовать TF-IDF

In [11]:
def tf_idf_vectorization(text: str, corpus: List[str] = None, vocab: List[str] = None, vocab_index: Dict[str, int] = None) -> List[float]:
    words_count = Counter(normalize_pretokenize_text(text))
    documents_words_count = [Counter(normalize_pretokenize_text(document)) for document in corpus]
    counts_in_documents = [sum(1 for doc_counter in documents_words_count if word in doc_counter) for word in words_count.keys()]
    result = [0.0] * len(vocab)
    for idx, (word, count) in enumerate(words_count.items()):
        if word not in vocab:
            continue
        result[vocab_index[word]] = (count / words_count.total()) * \
                            (math.log((len(corpus) + 1) / (counts_in_documents[idx] + 1)))    
    return result
    
def test_tf_idf_vectorization(corpus, vocab, vocab_index) -> bool:
    try:
        text = "the quick brown"
        result = tf_idf_vectorization(text, corpus, vocab, vocab_index)
        if not isinstance(result, list):
            return False

        expected_length = len(vocab)
        if len(result) != expected_length:
            return False

        for val in result:
            if not isinstance(val, float):
                return False

        print("TF-IDF test PASSED")
        return True
    except Exception as e:
        print(f"TF-IDF test FAILED: {e}")
        return False

In [12]:
assert test_tf_idf_vectorization(test_corpus, vocab, vocab_index)

TF-IDF test PASSED


## Задание 4 (1 балл)
Реализовать Positive Pointwise Mutual Information (PPMI).  
https://en.wikipedia.org/wiki/Pointwise_mutual_information
$$PPMI(word, context) = max(0, PMI(word, context))$$
$$PMI(word, context) = log \frac{P(word, context)}{P(word) P(context)} = log \frac{N(word, context)|(word, context)|}{N(word) N(context)}$$
где $N(word, context)$ -- число вхождений слова $word$ в окно $context$ (размер окна -- гиперпараметр)

In [13]:
def ppmi_vectorization(
    text: str,
    corpus: List[str] = None,
    vocab: List[str] = None,
    vocab_index: Dict[str, int] = None,
    window_size: int = 2
) -> List[float]:
    words = normalize_pretokenize_text(text)
    
    documents_words_count = Counter()
    window_context_counts = defaultdict(Counter)
    
    for doc in corpus:
        doc_words = normalize_pretokenize_text(doc)
        documents_words_count.update(doc_words)
        for i, word1 in enumerate(doc_words):
            start = max(i - window_size, 0)
            end = min(i + window_size + 1, len(doc_words))
            for j in range(start, end):
                if i != j:
                    window_context_counts[word1][doc_words[j]] += 1
    
    total_tokens = documents_words_count.total()
    total_context = sum(item.total() for item in window_context_counts.values())
    
    result = [0.0] * len(vocab)
    for word in words:
        if word not in vocab:
            continue
        ppmi_sum = 0
        for context_word, counts in window_context_counts[word].items():
            pw = documents_words_count[word] / total_tokens
            pc = documents_words_count[context_word] / total_tokens
            pwc = counts / total_context
            pmi = math.log(pwc / (pw * pc + 1e-8))
            ppmi = max(0, pmi)
            ppmi_sum += ppmi
        result[vocab_index[word]] = ppmi_sum
    return result

def test_ppmi_vectorization(corpus, vocab, vocab_index) -> bool:
    try:
        text = "quick brown fox"
        result = ppmi_vectorization(text, corpus, vocab, vocab_index)

        if not isinstance(result, list):
            return False

        expected_length = len(vocab)
        if len(result) != expected_length:
            return False

        for val in result:
            if not isinstance(val, float):
                return False

        print("PPMI test PASSED")
        return True
    except Exception as e:
        print(f"PPMI test FAILED: {e}")
        return False

In [14]:
assert test_ppmi_vectorization(test_corpus, vocab, vocab_index)

PPMI test PASSED


## Задание 5 (1 балл)
Реализовать получение эмбеддингов из fasttext и bert (для bert лучше использовать CLS токен)

In [15]:
# !wget https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.bin.gz

In [16]:
# !gunzip cc.en.300.bin.gz

In [17]:
# !rm cc.en.300.bin.gz

In [18]:
def get_fasttext_embeddings(text: str, model_path: str = None, model: any = None) -> List[np.ndarray]:
    if model_path is None:
        model_path = r"cc.en.300.bin"
    if model is None:
        model = fasttext.load_model(model_path)
    return [model.get_word_vector(x) for x in normalize_pretokenize_text(text)]
    # return model.get_sentence_vector(text)

In [19]:
get_fasttext_embeddings("quick brown fox")

[array([ 0.02726553, -0.10176626, -0.00404413,  0.09996106, -0.01571898,
         0.04248569,  0.2073496 ,  0.0087942 , -0.03639914,  0.00647714,
         0.06384429,  0.07365575,  0.06052569,  0.09319542,  0.01747438,
         0.10222745,  0.04015287, -0.04139482,  0.04251796,  0.01454664,
        -0.03042826,  0.00366084, -0.02781409,  0.0133833 , -0.07660622,
        -0.02829335,  0.05362656,  0.01393907, -0.00974267,  0.01524175,
        -0.06859868, -0.01214927, -0.0870764 , -0.01884183,  0.04669119,
        -0.00323973, -0.01214865,  0.00729405, -0.01671829,  0.0223159 ,
        -0.03287907, -0.04704074,  0.00924372,  0.07564097,  0.02308894,
         0.22239473, -0.07354219,  0.00677742, -0.03897931, -0.0066923 ,
        -0.02401984, -0.00171614,  0.09846944,  0.01784112, -0.04556665,
         0.04260909, -0.03917933,  0.08568592, -0.05624855,  0.02018312,
        -0.00297635,  0.01065139, -0.04050962, -0.07184873,  0.09240421,
         0.05162618, -0.1122213 , -0.06686787, -0.0

In [20]:
def get_bert_embeddings(
    text: str,
    model_name: str = 'bert-base-uncased',
    pool_method: str = 'cls'
) -> np.ndarray:
    
    bert_tokenizer = BertTokenizer.from_pretrained(model_name)
    bert_model = BertModel.from_pretrained(model_name)
    
    tokens = bert_tokenizer(text, return_tensors='pt', truncation=True, padding=True)
    bert_model.eval()
    with torch.no_grad():
        outputs = bert_model(**tokens)
    
    if pool_method == 'cls':
        return outputs.last_hidden_state[:, 0, :].squeeze(0).cpu().numpy()

    

In [21]:
get_bert_embeddings("quick brown fox")

array([-7.73346424e-01, -1.52948618e-01, -3.96935463e-01,  3.66077811e-01,
        9.68404040e-02,  2.99831927e-01, -1.80257425e-01,  4.50202972e-01,
       -6.11764610e-01, -8.78362209e-02, -1.57348543e-01,  1.83511063e-01,
        5.23932874e-02,  5.23251414e-01, -7.05117658e-02, -2.15942264e-01,
       -5.21603405e-01,  4.96029586e-01,  2.92461272e-02, -6.66669980e-02,
        2.59652585e-01,  6.21663369e-02, -4.24834102e-01, -2.75132447e-01,
        2.66273916e-01,  2.59304076e-01,  2.01495588e-02,  4.24167439e-02,
       -1.15836538e-01,  4.44841295e-01, -6.88874573e-02, -3.51457559e-02,
       -1.53637063e-02,  2.10701823e-01,  1.32001549e-01, -2.54509121e-01,
        3.67906272e-01, -4.36001569e-02, -1.01087958e-01, -8.18075985e-03,
        1.89969331e-01,  1.13654755e-01,  3.34827483e-01, -1.27592102e-01,
       -1.67596966e-01, -1.39888912e-01, -2.83615255e+00, -2.15618342e-01,
       -9.69092697e-02, -6.97880030e-01, -3.54389220e-01, -1.10328063e-01,
        2.13751316e-01,  

## Задание 6 (1.5 балла)
Реализовать обучение так, чтобы можно было поверх эмбеддингов, реализованных в предыдущих заданиях, обучить какую-то модель (вероятно неглубокую, например, CatBoost) на задаче классификации текстов ([IMDB](https://huggingface.co/datasets/stanfordnlp/imdb)).

In [ ]:
def vectorize_dataset(
    dataset_name: str = "imdb",
    vectorizer_type: str = "bow",
    split: str = "train",
    sample_size: int = 2500
) -> Tuple[Any, List, List]:

    dataset = datasets.load_dataset(dataset_name, split=split)
    dataset = dataset.shuffle(seed=42)
    dataset = dataset.select(range(min(sample_size, len(dataset))))

    texts = [item['text'] for item in dataset if 'text' in item and item['text'].strip()]
    labels = [item['label'] for item in dataset if 'label' in item]

    def build_vocab(texts: List[str]) -> Tuple[List[str], Dict[str, int]]:
        all_words = []
        for text in texts:
            words = normalize_pretokenize_text(text)
            all_words.extend(words)
        vocab = sorted(set(all_words))
        vocab_index = {word: idx for idx, word in enumerate(vocab)}
        return vocab, vocab_index

    global vocab, vocab_index
    if vocab is None or vocab_index is None:
        vocab, vocab_index = build_vocab(texts)

    vectorized_data = []
    for text in texts:
        if vectorizer_type == "one_hot":
            vectorized_data.append(one_hot_vectorization(text, vocab, vocab_index))
        elif vectorizer_type == "bow":
            bow_dict = bag_of_words_vectorization(text)
            vector = [bow_dict.get(word, 0) for word in vocab]
            vectorized_data.append(vector)
        elif vectorizer_type == "tfidf":
            vectorized_data.append(tf_idf_vectorization(text, texts, vocab, vocab_index)) # Ассимптотика N^3 минимум с базовой реалзиацией. Честно ждать пока всё посчитаеться лень
        elif vectorizer_type == "ppmi":
            vectorized_data.append(ppmi_vectorization(text, texts, vocab, vocab_index)) # Ассимптотика намного хуже...
        elif vectorizer_type == "fasttext":
            embeddings = get_fasttext_embeddings(text)
            if embeddings:
                avg_embedding = np.mean(embeddings, axis=0)
                vectorized_data.append(avg_embedding.tolist())
            else:
                vectorized_data.append([0] * 300)
        elif vectorizer_type == "bert":
            embedding = get_bert_embeddings(text)
            vectorized_data.append(embedding.tolist())
        else:
            raise ValueError(f"Unknown vectorizer type: {vectorizer_type}")
    return vocab, vectorized_data, labels

In [23]:
# from catboost import CatBoostClassifier
# from sklearn.metrics import classification_report, accuracy_score, f1_score
# from sklearn.model_selection import train_test_split, cross_val_score, KFold

# def train(
#     embeddings_method="bow",
#     test_size=0.2,
#     val_size=0.2,
#     cv_folds=5
# ):
#     _, X, y = vectorize_dataset("imdb", embeddings_method, "train")
#     _, X_test, y_test = vectorize_dataset("imdb", embeddings_method, "test")

#     model = CatBoostClassifier(
#         iterations=100,
#         learning_rate=1e-3,
#         depth=5,
#         random_state=42
#     )
#     model.fit(X = X, y = y)
#     print(classification_report(y_test, model.predict(X_test)))

In [24]:
# for embeddings_method in ["bow", "one_hot", "tfidf", "ppmi", "fasttext", "bert"]:
#     vocab = None
#     vocab_index = None
#     train(embeddings_method=embeddings_method)

## Моя реализация того же самого
Оптимизации для усорения расчётов в основном

In [25]:
def getData(
    dataset_name: str = "imdb",
    split: str = "train",
    sample_size: int = 2500
) -> Tuple[Any, List, List]:
    dataset = datasets.load_dataset(dataset_name, split=split)
    dataset = dataset.shuffle(seed=42)
    dataset = dataset.select(range(min(sample_size, len(dataset))))
    
    texts = [item['text'] for item in dataset if 'text' in item and item['text'].strip()]
    labels = [item['label'] for item in dataset if 'label' in item]

    return texts, labels

In [26]:
texts, labels = getData() # random_state = 42
print(texts[:5])
print(labels[:5])

['There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. Fortier\'s plot are far more complicated... Fortier looks more like Prime Suspect, if we have to spot similarities... The main character is weak and weirdo, but have "clairvoyance". People like to compare, to judge, to evaluate. How about just enjoying? Funny thing too, people writing Fortier looks American but, on the other hand, arguing they prefer American series (!!!). Maybe it\'s the language, or the spirit, but I think this series is more English than American. By the way, the actors are really good and funny. The acting is not superficial at all...', 'This movie is a great. The plot is very true to the book which is a classic written by Mark Twain. The movie starts of with a scene where Hank sings a song with a bunch of kids called "when you stub your toe on the moon" It reminds me

In [27]:
def build_vocab_one_hot(texts: List[str]) -> Tuple[List[str], Dict[str, int]]:
    counter = Counter()
    for text in texts:
        words = normalize_pretokenize_text(text)
        counter.update(words)
    vocab = [word for idx, (word, _) in enumerate(counter.most_common()) if idx < 1000]
    vocab.append('[NOEXIST]')
    vocab_index = {word: idx for idx, word in enumerate(vocab)}
    return vocab, vocab_index

vocab, vocab_index = build_vocab_one_hot(texts)

In [28]:
def one_hot_vectorization(text: str, vocab: List[str] = None, vocab_index: Dict[str, int] = None) -> List[int]:
    words = normalize_pretokenize_text(text)
    one_hot_matrix = [0] * len(vocab)
    for word in words:
        one_hot_matrix[vocab_index.get(word, len(vocab) - 1)] = 1
    return one_hot_matrix

In [29]:
one_hot_vocab = None
one_hot_vocab_index = None

In [30]:
def vectorization_one_hot(texts, split: str = "train"):
    global one_hot_vocab, one_hot_vocab_index
    if split == "train":
        embeddings = []
        one_hot_vocab, one_hot_vocab_index = build_vocab_one_hot(texts)
        for text in texts:
            embeddings.append(one_hot_vectorization(text, one_hot_vocab, one_hot_vocab_index))
        return embeddings
    elif split == "test":
        if one_hot_vocab is None or one_hot_vocab_index is None:
            raise AttributeError("One hot vocab is not exists") 
        embeddings = []
        for text in texts:
            embeddings.append(one_hot_vectorization(text, one_hot_vocab, one_hot_vocab_index))
        return embeddings

In [31]:
bag_of_words_vocab = None
bag_of_words_vocab_index = None

def vectorization_bag_of_words(texts, split: str = "train"):
    global bag_of_words_vocab, bag_of_words_vocab_index
    if split == "train":
        bag_of_words_vocab, bag_of_words_vocab_index = build_vocab(texts)
    embedding = []
    for text in texts:
        bow_dict = bag_of_words_vectorization(text)
        vector = [bow_dict.get(word, 0) for word in bag_of_words_vocab]
        embedding.append(vector)
    return embedding

In [32]:
tf_idf_words_vocab = None
tf_idf_words_vocab_index = None
documents_word_counter = None

def vectorization_tf_idf(texts, split: str = "train"):
    global tf_idf_words_vocab, tf_idf_words_vocab_index, documents_word_counter
    if split == "train":
        tf_idf_words_vocab, tf_idf_words_vocab_index = build_vocab(texts)
        documents_word_counter = Counter()
        for text in texts:
            words = set(normalize_pretokenize_text(text))
            documents_word_counter.update(words)
    
    tf_idf_matrix = np.zeros((len(texts), len(tf_idf_words_vocab)))
    for i, text in enumerate(texts):
        words = normalize_pretokenize_text(text)
        tf_counter = Counter(words)

        for word, count in tf_counter.items():
            if word in tf_idf_words_vocab_index:
                tf = count / len(words)
                idf = math.log(len(texts) / (1 + documents_word_counter[word]))
                tf_idf_matrix[i, tf_idf_words_vocab_index[word]] = tf * idf
    return tf_idf_matrix

$$PPMI(word, context) = max(0, PMI(word, context))$$
$$PMI(word, context) = log \frac{P(word, context)}{P(word) P(context)} = log \frac{N(word, context)|(word, context)|}{N(word) N(context)}$$
где $N(word, context)$ -- число вхождений слова $word$ в окно $context$ (размер окна -- гиперпараметр)

In [ ]:
ppmi_documents_words_count = None
ppmi_window_context_counts = None
ppmi_vocab = None
ppmi_vocab_index = None
ppmi_values = None

def vectorize_ppmi(texts, split="train", window_size = 2):
    global ppmi_documents_words_count, ppmi_window_context_counts, ppmi_vocab, ppmi_vocab_index, ppmi_values
    if split == "train":
        ppmi_documents_words_count = Counter()
        ppmi_window_context_counts = defaultdict(Counter)
        for text in texts:
            words = normalize_pretokenize_text(text)
            ppmi_documents_words_count.update(words)
            for i, word1 in enumerate(words):
                start = max(0, i - window_size)
                end = min(len(words), i + window_size + 1)
                for j in range(start, end):
                    if i != j:
                        ppmi_window_context_counts[word1][words[j]] += 1
        ppmi_vocab, ppmi_vocab_index = build_vocab(texts)
        total_tokens = ppmi_documents_words_count.total()
        total_context = sum(item.total() for item in ppmi_window_context_counts.values())

        ppmi_values = defaultdict(dict)

        for word, contexts in ppmi_window_context_counts.items():
            pw = ppmi_documents_words_count[word] / total_tokens
            for context_word, counts in contexts.items():
                pc = ppmi_documents_words_count[context_word] / total_tokens
                pwc = counts / total_context
                pmi = math.log(pwc / (pw * pc + 1e-8))
                ppmi_values[word][context_word] = max(0, pmi)
    
    result = np.zeros((len(texts), len(ppmi_vocab)))
    for idx, text in enumerate(texts):
        for word in normalize_pretokenize_text(text):
            if word in ppmi_vocab_index and word in ppmi_values:
                ppmi_sum = sum(ppmi_values[word].values())
                result[idx][ppmi_vocab_index[word]] = ppmi_sum
    return result
        

In [33]:
def vectorize_fasttext(texts, split: str = "train", model_path = None, model = None):
    if model_path is None and model is None:
        model_path = r"cc.en.300.bin"
    if model is None:
        model = fasttext.load_model(model_path)
    result = []
    for text in texts:
        words = normalize_pretokenize_text(text)
        if words:
            word_vectors = np.array([model.get_word_vector(w) for w in words])
            avg_vector = np.mean(word_vectors, axis=0)
        else:
            avg_vector = np.zeros(300)
        result.append(avg_vector.tolist())
    return result

In [41]:
def vectorize_bert(texts, split="train", model_name = 'bert-base-uncased', pool_method = 'cls'):
    result = []
    
    bert_tokenizer = BertTokenizer.from_pretrained(model_name)
    bert_model = BertModel.from_pretrained(model_name)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    bert_model.to(device)
    bert_model.eval()
    for text in texts:
        tokens = bert_tokenizer(text, return_tensors='pt', truncation=True, padding=True)
        tokens.to(device)
        with torch.no_grad():
            outputs = bert_model(**tokens)
        if pool_method == 'cls':
            result.append(outputs.last_hidden_state[:, 0, :].squeeze(0).cpu().numpy())
    return result

In [45]:
def vectorize_dataset(
    dataset_name: str = "imdb",
    vectorizer_type: str = "bow",
    split: str = "train",
    sample_size: int = 2500
) -> Tuple[Any, List, List]:

    texts, labels = getData(dataset_name=dataset_name, split=split, sample_size=sample_size)

    
    if vectorizer_type == "one_hot":
        vectorized_data = vectorization_one_hot(texts, split=split)
    elif vectorizer_type == "bow":
        vectorized_data = vectorization_bag_of_words(texts, split=split)
    elif vectorizer_type == "tfidf":
        vectorized_data = vectorization_tf_idf(texts, split=split)
    elif vectorizer_type == "ppmi":
        vectorized_data = vectorize_ppmi(texts, split=split)
    elif vectorizer_type == "fasttext":
        vectorized_data = vectorize_fasttext(texts, split=split)
    elif vectorizer_type == "bert":
        vectorized_data = vectorize_bert(texts, split=split)
    else:
        raise ValueError(f"Unknown vectorizer type: {vectorizer_type}")
    
    return vocab, vectorized_data, labels

In [54]:
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.model_selection import train_test_split, cross_val_score, KFold

def train(
    embeddings_method="bow",
    test_size=0.2,
    val_size=0.2,
    cv_folds=5
):
    _, X, y = vectorize_dataset("imdb", embeddings_method, "train")
    _, X_test, y_test = vectorize_dataset("imdb", embeddings_method, "test")

    X_train_sub, X_val, y_train_sub, y_val = train_test_split(X, y, test_size=val_size, random_state=42)

    model = CatBoostClassifier(
        iterations=4000,
        learning_rate=1e-2,
        depth=7,
        random_state=42,
        early_stopping_rounds=20,
    )
    model.fit(X = X_train_sub, y = y_train_sub,
            eval_set=(X_val, y_val),
            verbose=400
    )
    print(classification_report(y_test, model.predict(X_test)))

In [55]:
for embeddings_method in ["one_hot", "bow", "tfidf", "ppmi", "fasttext", "bert"]:
    train(embeddings_method=embeddings_method)

0:	learn: 0.6904816	test: 0.6912617	best: 0.6912617 (0)	total: 24.5ms	remaining: 1m 37s
400:	learn: 0.4096845	test: 0.4987265	best: 0.4987265 (400)	total: 10.3s	remaining: 1m 32s
800:	learn: 0.3078451	test: 0.4465182	best: 0.4465182 (800)	total: 20.4s	remaining: 1m 21s
1200:	learn: 0.2178875	test: 0.4144016	best: 0.4144016 (1200)	total: 30.7s	remaining: 1m 11s
1600:	learn: 0.1573215	test: 0.3977033	best: 0.3977033 (1600)	total: 40.9s	remaining: 1m 1s
Stopped by overfitting detector  (20 iterations wait)

bestTest = 0.3915633577
bestIteration = 1902

Shrink model to first 1903 iterations.
              precision    recall  f1-score   support

           0       0.85      0.80      0.82      1243
           1       0.81      0.86      0.83      1257

    accuracy                           0.83      2500
   macro avg       0.83      0.83      0.83      2500
weighted avg       0.83      0.83      0.83      2500

0:	learn: 0.6910914	test: 0.6918200	best: 0.6918200 (0)	total: 57.8ms	remainin

```text
One-hot-encoding
Shrink model to first 1903 iterations.
              precision    recall  f1-score   support

           0       0.85      0.80      0.82      1243
           1       0.81      0.86      0.83      1257

    accuracy                           0.83      2500
   macro avg       0.83      0.83      0.83      2500
weighted avg       0.83      0.83      0.83      2500
```

```text
bag of words
Shrink model to first 2274 iterations.
              precision    recall  f1-score   support

           0       0.86      0.77      0.81      1243
           1       0.79      0.88      0.83      1257

    accuracy                           0.82      2500
   macro avg       0.83      0.82      0.82      2500
weighted avg       0.83      0.82      0.82      2500
```

```text
tf-idf
Shrink model to first 1514 iterations.
              precision    recall  f1-score   support

           0       0.86      0.78      0.82      1243
           1       0.80      0.87      0.83      1257

    accuracy                           0.83      2500
   macro avg       0.83      0.83      0.83      2500
weighted avg       0.83      0.83      0.83      2500
```

```text
ppmi
Shrink model to first 2810 iterations.
              precision    recall  f1-score   support

           0       0.87      0.79      0.83      1243
           1       0.81      0.88      0.84      1257

    accuracy                           0.83      2500
   macro avg       0.84      0.83      0.83      2500
weighted avg       0.84      0.83      0.83      2500
```

```text
fasttext
Shrink model to first 1104 iterations.
              precision    recall  f1-score   support

           0       0.77      0.75      0.76      1243
           1       0.76      0.77      0.77      1257

    accuracy                           0.76      2500
   macro avg       0.76      0.76      0.76      2500
weighted avg       0.76      0.76      0.76      2500
```

```text
bert
Shrink model to first 1167 iterations.
              precision    recall  f1-score   support

           0       0.83      0.80      0.82      1243
           1       0.81      0.84      0.82      1257

    accuracy                           0.82      2500
   macro avg       0.82      0.82      0.82      2500
weighted avg       0.82      0.82      0.82      2500
```